# 🦠 JORINOVA NEXUS — Fungal (DeFungi) CLASSIFIER (fine-tuning)

Classifies fungal elements (Entamoeba histolytica, Giardia, Cryptosporidium) on stool
microscopy. This is a **classification** model (`yolov8*-cls`) — the ready public data (Kaggle
fungal-parasite image set) is class-labelled, and classification reaches high accuracy
(metric = **top-1 accuracy**).

Produces `mycology.pt`; the predicted class resolves to its disease via
`backend/ai_services/mycology_organisms.json`.

> **Fine-tuning, NOT from scratch** (starts from `yolov8m-cls.pt`).
> **Data source: KAGGLE** — upload your `kaggle.json` in step 4.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_mycology', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_mycology')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (KAGGLE — fungal parasite image set)
Upload your `kaggle.json` (kaggle.com → Settings → API → **Create New Token**). The cell downloads
the **fungal-parasite** set (Cryptosporidium / Entamoeba / Giardia), auto-detects the class
folders, and renames them to the app keys in `mycology_organisms.json`.

In [ ]:
# -- 4. Mycology data via KAGGLE (DeFungi classification) -> train/val class folders --
# DeFungi = direct microscopic fungal examination (Candida, Aspergillus, dermatophytes).
# Class folders are auto-detected and canon-mapped to app keys in mycology_organisms.json.
import os, glob, shutil, subprocess, sys, random
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload kaggle.json (kaggle.com -> Settings -> API -> Create New Token) ...'); up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    name = 'kaggle.json' if os.path.exists('kaggle.json') else list(up)[0]
    shutil.move(name, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

RAW = '/content/data/mycology_raw'
if os.path.exists(RAW): shutil.rmtree(RAW)
os.makedirs(RAW, exist_ok=True)
SLUGS = ['joebeachcapital/defungi', 'sajjadhussainkhan/defungi', 'jderobles/defungi']
ok = False
for DATASET in SLUGS:
    rc = subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET, '-p', RAW, '--unzip']).returncode
    if rc == 0: ok = True; print('downloaded:', DATASET); break
    print('failed:', DATASET)
assert ok, 'Kaggle download failed - accept the DeFungi rules on kaggle.com + check kaggle.json. Search kaggle.com/datasets?search=defungi and add the slug to SLUGS.'

def canon(n):
    n = n.lower().strip()
    m = {'h1':'candida_albicans','h2':'aspergillus_niger','h3':'trichophyton_mentagrophytes','h5':'trichophyton_rubrum','h6':'epidermophyton_floccosum'}
    if n in m: return m[n]
    if 'candida' in n: return 'candida_albicans'
    if 'aspergill' in n: return 'aspergillus_niger'
    if 'mentagro' in n: return 'trichophyton_mentagrophytes'
    if 'rubrum' in n: return 'trichophyton_rubrum'
    if 'epidermo' in n: return 'epidermophyton_floccosum'
    if 'trichophyton' in n: return 'trichophyton_mentagrophytes'
    return n.replace(' ', '_').replace('-', '_')

exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.tif','*.tiff')
CLS = '/content/data/mycology_cls'
if os.path.exists(CLS): shutil.rmtree(CLS)
n_imgs = 0
for d, subs, files in os.walk(RAW):
    imgs = [p for e in exts for p in glob.glob(os.path.join(d, e))]
    if not imgs: continue
    key = canon(os.path.basename(d))
    random.shuffle(imgs); nval = max(1, len(imgs)//6)
    for i, ip in enumerate(imgs):
        split = 'val' if i < nval else 'train'
        out = os.path.join(CLS, split, key); os.makedirs(out, exist_ok=True)
        shutil.copy(ip, os.path.join(out, '%s_%d%s' % (key, n_imgs+i, os.path.splitext(ip)[1])))
    n_imgs += len(imgs)
assert n_imgs, 'No images found under ' + RAW + ' - print os.listdir(RAW) and send it to me.'
DATA_DIR = CLS
print('DATA_DIR =', DATA_DIR)
print('classes:', sorted(os.listdir(os.path.join(CLS, 'train'))))

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8m-cls.pt'   # classification model (yolov8s-cls = faster/less accurate)
print('fine-tuning (classification) from:', BASE_WEIGHTS)

## 6. Fine-tune the classifier
Metric is **top-1 accuracy** (not mAP). Mycology cyst/oocyst crops → `imgsz=224`, `yolov8m-cls`,
~120 epochs, cosine LR + dropout/erasing. Microscopy has no canonical orientation → flips + rotation are safe.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_mycology/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/classify'
model = YOLO(BASE_WEIGHTS)   # yolov8m-cls -> classification (mycology cyst/oocyst crops)
model.train(data=DATA_DIR, epochs=120, imgsz=224, batch=64,
            project=PROJECT, name='mycology', pretrained=True, patience=30, plots=True,
            cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 6b. RESUME if a run disconnects (continue, no 4-vs-80 crash)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_mycology/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_mycology/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_DIR' in globals(), 'Re-run step 4 first so DATA_DIR is set.'
    model = YOLO(ckpt); model.train(data=DATA_DIR, epochs=120, imgsz=224, batch=64,
        project='/content/drive/MyDrive/nexus_mycology/runs', name='mycology_cont', pretrained=True, patience=30,
        cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
from ultralytics import YOLO
DRIVE = '/content/drive/MyDrive/nexus_mycology'
cands = sorted(glob.glob(DRIVE + '/runs/**/weights/best.pt', recursive=True), key=os.path.getmtime)
assert cands, 'No mycology run in ' + DRIVE + '/runs/. Re-run TRAIN (step 6); ensure DATASET (step 4) set DATA_DIR.'
best = cands[-1]
names = list(YOLO(best).names.values())
assert not {'person','bicycle','car'}.issubset(set(names)), \
    ('Picked weights have COCO classes ' + str(names[:4]) + ' -> training did NOT use the DeFungi dataset. Re-run step 4 then step 6.')
print('using weights:', best); print('classes:', names)
os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/mycology.pt'); print('saved ->', DRIVE + '/mycology.pt')
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`mycology.pt`** at **`backend/models/mycology/mycology.pt`**, commit, push.
2. Auto-loads for `image_type` ∈ {`mycology`, `stool_mycology`}; the predicted class resolves via `mycology_organisms.json`. The vision service **handles classification models automatically** (top-1 + top-k).
3. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the field.

> ⚠️ Screening aid — a microscopist confirms (wet prep / trichrome / modified acid-fast).